# Deep_GLOC masked-seed split generator

这个 notebook 用于为 **Deep_GLOC / RWR / DeepWalk** 生成统一的 masked-seed benchmark split 方案，并额外生成 **Deep_GLOC 专用的无泄漏 PyG bundle**。

## 目的
1. 对每个过程的核心基因做重复随机遮挡；
2. 输出统一的 `train_seeds` / `heldout_seeds` 文件，供你和师弟共用；
3. 为 Deep_GLOC 额外生成每个 split 对应的 `.pt` 文件：
   - 更新 `supervised_mask`
   - 更新 `y`
   - 对 held-out 节点清除 core/process 特征，避免信息泄漏

## 为什么必须清除 held-out 特征
你当前的 `x` 特征里包含：
- `is_core_gene`
- `is_native_core`
- `is_anchored_core`
- `proc_Biosynthesis`
- `proc_Esterification`
- `proc_Excretion`
- `proc_Uptake`

如果只是把 held-out 基因从 `train_mask/supervised_mask` 里拿掉，但仍保留这些特征，模型实际上仍然“知道”这些基因是哪个过程的 core gene，这会导致 benchmark 不公平。


In [1]:
import os
import copy
import random
import numpy as np
import pandas as pd
import torch

# ==============================
# 路径配置（按你本地实际路径修改）
# ==============================
INPUT_PYG_FILE = r"D:\phd cases\metabolic analysis\code\Deep_GLOC\run_GAT_new\augmented_graph_output\pyg_data_with_topology_features.pt"

OUT_DIR = r"D:\phd cases\metabolic analysis\code\Deep_GLOC\run_GAT_new\masked_seed_benchmark"
SPLIT_DIR = os.path.join(OUT_DIR, "splits")
PYG_SPLIT_DIR = os.path.join(OUT_DIR, "pyg_split_bundles")
SUMMARY_DIR = os.path.join(OUT_DIR, "summary")

for d in [OUT_DIR, SPLIT_DIR, PYG_SPLIT_DIR, SUMMARY_DIR]:
    os.makedirs(d, exist_ok=True)

# ==============================
# split 参数
# ==============================
BASE_SEED = 2026
N_REPEATS = 100
HOLDOUT_FRAC = 0.30
MIN_HOLDOUT = 1

# 如果某过程 hard seeds 太少，则自动切换 LOOCV
# 例如 n_hard <= 8 时，每次只留 1 个
LOOCV_THRESHOLD = 8

print("OUT_DIR =", OUT_DIR)


OUT_DIR = D:\phd cases\metabolic analysis\code\Deep_GLOC\run_GAT_new\masked_seed_benchmark


In [2]:
# ==============================
# 读取 bundle
# ==============================
bundle = torch.load(INPUT_PYG_FILE, map_location="cpu", weights_only=False)

data = bundle["data"]
node_df = bundle["node_df"].copy()
edge_df = bundle["edge_df"].copy()
hard_seed_df = bundle.get("hard_seed_df", pd.DataFrame()).copy()
ambiguous_seed_df = bundle.get("ambiguous_seed_df", pd.DataFrame()).copy()
core_all = bundle.get("core_all", pd.DataFrame()).copy()
aux_info = bundle.get("aux_info", {})

print(data)
print("node_df shape:", node_df.shape)
print("edge_df shape:", edge_df.shape)
print("hard_seed_df shape:", hard_seed_df.shape)
print("ambiguous_seed_df shape:", ambiguous_seed_df.shape)
print("core_all shape:", core_all.shape)

if "process_map" in aux_info:
    process_map = aux_info["process_map"]
else:
    process_map = {
        "Biosynthesis": 0,
        "Esterification": 1,
        "Excretion": 2,
        "Uptake": 3
    }

label2process = {v: k for k, v in process_map.items()}
process_order = [label2process[i] for i in sorted(label2process)]

if "gene2idx" in aux_info:
    gene2idx = aux_info["gene2idx"]
else:
    if "ENSGID" not in node_df.columns:
        raise ValueError("node_df 中缺少 ENSGID，且 aux_info 中也没有 gene2idx")
    gene2idx = {g: i for i, g in enumerate(node_df["ENSGID"].astype(str).tolist())}

if "feature_cols" in aux_info:
    feature_cols = list(aux_info["feature_cols"])
else:
    feature_cols = None

print("process_order =", process_order)
print("feature_cols =", feature_cols)


Data(x=[3211, 19], edge_index=[2, 693282], edge_attr=[693282, 3], y=[3211], edge_weight=[693282], supervised_mask=[3211], train_mask=[3211], all_core_mask=[3211], hard_seed_mask=[3211], ambiguous_core_mask=[3211], seed_weight=[3211], edge_type=[693282], seed_mask_Biosynthesis=[3211], seed_mask_Esterification=[3211], seed_mask_Excretion=[3211], seed_mask_Uptake=[3211])
node_df shape: (3211, 46)
edge_df shape: (346641, 12)
hard_seed_df shape: (39, 5)
ambiguous_seed_df shape: (0, 5)
core_all shape: (39, 4)
process_order = ['Biosynthesis', 'Esterification', 'Excretion', 'Uptake']
feature_cols = ['is_in_meta', 'degree_z', 'weighted_degree_z', 'meta_strength_z', 'ppi_strength_z', 'ppi_ratio', 'has_ppi_edge', 'seed_weight_sum_Biosynthesis_z', 'seed_weight_sum_Esterification_z', 'seed_weight_sum_Excretion_z', 'seed_weight_sum_Uptake_z', 'avg_dist_Biosynthesis_z', 'avg_dist_Esterification_z', 'avg_dist_Excretion_z', 'avg_dist_Uptake_z', 'diffusion_Biosynthesis_z', 'diffusion_Esterification_z', 

## 只用 hard seeds 生成 masked-seed splits

主 benchmark 建议只对 **hard seeds（单一过程标签）** 做遮挡，因为：
1. 它们对应唯一标签，适合做 process-specific recovery；
2. Deep_GLOC 当前监督训练本身也是基于 hard seeds；
3. ambiguous seeds 属于多过程标签，不适合作为单标签 held-out positives。


In [3]:
# ==============================
# 预处理 hard seeds
# ==============================
required_cols = ["ENSGID", "process", "label"]
for c in required_cols:
    if c not in hard_seed_df.columns:
        raise ValueError(f"hard_seed_df 缺少必要列: {c}")

hard_seed_df = hard_seed_df.copy()
hard_seed_df["ENSGID"] = hard_seed_df["ENSGID"].astype(str).str.strip().str.split(".").str[0]
hard_seed_df["process"] = hard_seed_df["process"].astype(str).str.strip()

# 去重：一个基因在 hard_seed_df 中理论上只属于一个 process
hard_seed_unique = hard_seed_df[["ENSGID", "process", "label"]].drop_duplicates().copy()

# 仅保留图中存在的 hard seeds
hard_seed_unique["in_graph"] = hard_seed_unique["ENSGID"].isin(gene2idx)
hard_seed_in_graph = hard_seed_unique[hard_seed_unique["in_graph"]].copy()

print("hard seeds total:", hard_seed_unique.shape[0])
print("hard seeds in graph:", hard_seed_in_graph.shape[0])

seed_count_df = (
    hard_seed_in_graph.groupby("process")["ENSGID"]
    .nunique()
    .reset_index(name="n_hard_in_graph")
    .sort_values("process")
    .reset_index(drop=True)
)
display(seed_count_df)


hard seeds total: 39
hard seeds in graph: 39


,process,n_hard_in_graph
0,Biosynthesis,15
1,Esterification,6
2,Excretion,10
3,Uptake,8


In [4]:
def make_process_splits(genes, process_name, n_repeats=100, holdout_frac=0.3,
                        min_holdout=1, loocv_threshold=8, base_seed=2026):
    """
    genes: list[str], 某个 process 的 hard seeds（已确保在图中）
    返回：list of dict
    """
    genes = sorted(set(genes))
    n = len(genes)
    splits = []

    if n == 0:
        return splits

    # 小基因集：LOOCV
    if n <= loocv_threshold:
        for i, g in enumerate(genes, start=1):
            heldout = [g]
            train = [x for x in genes if x != g]
            splits.append({
                "process": process_name,
                "repeat_id": i,
                "scheme": "LOOCV",
                "n_total": n,
                "n_train": len(train),
                "n_heldout": len(heldout),
                "train_genes": train,
                "heldout_genes": heldout
            })
        return splits

    # 普通 repeated random holdout
    rng = random.Random(base_seed + abs(hash(process_name)) % 100000)
    for i in range(1, n_repeats + 1):
        n_heldout = max(min_holdout, int(round(n * holdout_frac)))
        n_heldout = min(n_heldout, n - 1)  # 至少保留 1 个 train seed

        genes_copy = genes.copy()
        rng.shuffle(genes_copy)

        heldout = sorted(genes_copy[:n_heldout])
        train = sorted(genes_copy[n_heldout:])

        splits.append({
            "process": process_name,
            "repeat_id": i,
            "scheme": "random_holdout",
            "n_total": n,
            "n_train": len(train),
            "n_heldout": len(heldout),
            "train_genes": train,
            "heldout_genes": heldout
        })

    return splits


In [5]:
# ==============================
# 生成 split 方案
# ==============================
all_split_rows = []
split_meta_rows = []

for proc in process_order:
    proc_genes = sorted(
        hard_seed_in_graph.loc[hard_seed_in_graph["process"] == proc, "ENSGID"].astype(str).unique().tolist()
    )

    proc_splits = make_process_splits(
        genes=proc_genes,
        process_name=proc,
        n_repeats=N_REPEATS,
        holdout_frac=HOLDOUT_FRAC,
        min_holdout=MIN_HOLDOUT,
        loocv_threshold=LOOCV_THRESHOLD,
        base_seed=BASE_SEED
    )

    for sp in proc_splits:
        split_id = f"{proc}__split_{sp['repeat_id']:03d}"
        split_meta_rows.append({
            "split_id": split_id,
            "process": proc,
            "repeat_id": sp["repeat_id"],
            "scheme": sp["scheme"],
            "n_total": sp["n_total"],
            "n_train": sp["n_train"],
            "n_heldout": sp["n_heldout"]
        })

        for g in sp["train_genes"]:
            all_split_rows.append({
                "split_id": split_id,
                "process": proc,
                "repeat_id": sp["repeat_id"],
                "scheme": sp["scheme"],
                "set_type": "train",
                "ENSGID": g
            })

        for g in sp["heldout_genes"]:
            all_split_rows.append({
                "split_id": split_id,
                "process": proc,
                "repeat_id": sp["repeat_id"],
                "scheme": sp["scheme"],
                "set_type": "heldout",
                "ENSGID": g
            })

split_long_df = pd.DataFrame(all_split_rows)
split_meta_df = pd.DataFrame(split_meta_rows)

print("split_long_df shape:", split_long_df.shape)
print("split_meta_df shape:", split_meta_df.shape)
display(split_meta_df.head())
display(split_long_df.head())


split_long_df shape: (2600, 6)
split_meta_df shape: (214, 7)


,split_id,process,repeat_id,scheme,n_total,n_train,n_heldout
0,Biosynthesis__split_001,Biosynthesis,1,random_holdout,15,11,4
1,Biosynthesis__split_002,Biosynthesis,2,random_holdout,15,11,4
2,Biosynthesis__split_003,Biosynthesis,3,random_holdout,15,11,4
3,Biosynthesis__split_004,Biosynthesis,4,random_holdout,15,11,4
4,Biosynthesis__split_005,Biosynthesis,5,random_holdout,15,11,4


,split_id,process,repeat_id,scheme,set_type,ENSGID
0,Biosynthesis__split_001,Biosynthesis,1,random_holdout,train,ENSG00000052802
1,Biosynthesis__split_001,Biosynthesis,1,random_holdout,train,ENSG00000104549
2,Biosynthesis__split_001,Biosynthesis,1,random_holdout,train,ENSG00000109929
3,Biosynthesis__split_001,Biosynthesis,1,random_holdout,train,ENSG00000110921
4,Biosynthesis__split_001,Biosynthesis,1,random_holdout,train,ENSG00000112972


In [6]:
# ==============================
# 保存 split 文件
# ==============================
split_long_path = os.path.join(SUMMARY_DIR, "masked_seed_splits_long.tsv")
split_meta_path = os.path.join(SUMMARY_DIR, "masked_seed_splits_meta.tsv")

split_long_df.to_csv(split_long_path, sep="\t", index=False)
split_meta_df.to_csv(split_meta_path, sep="\t", index=False)

# 同时按 split 输出 train / heldout 文件
for split_id, sub in split_long_df.groupby("split_id"):
    out_subdir = os.path.join(SPLIT_DIR, split_id)
    os.makedirs(out_subdir, exist_ok=True)

    train_df = sub[sub["set_type"] == "train"][["ENSGID"]].drop_duplicates().copy()
    held_df = sub[sub["set_type"] == "heldout"][["ENSGID"]].drop_duplicates().copy()

    train_df.to_csv(os.path.join(out_subdir, "train_seeds.tsv"), sep="\t", index=False)
    held_df.to_csv(os.path.join(out_subdir, "heldout_seeds.tsv"), sep="\t", index=False)

print("saved:")
print(split_long_path)
print(split_meta_path)


saved:
D:\phd cases\metabolic analysis\code\Deep_GLOC\run_GAT_new\masked_seed_benchmark\summary\masked_seed_splits_long.tsv
D:\phd cases\metabolic analysis\code\Deep_GLOC\run_GAT_new\masked_seed_benchmark\summary\masked_seed_splits_meta.tsv


## 生成 Deep_GLOC 专用 no-leak bundle

这里会为每个 split 复制一个新的 PyG bundle，并做 3 件事：

1. 将 held-out genes 从 `supervised_mask` 中移除；
2. 将 held-out genes 的 `y` 改成 `-1`；
3. 将 held-out genes 的 core/process 特征清零，避免信息泄漏。

### 会被清零的特征
如果 `feature_cols` 存在，则按列名自动识别这些字段：
- `is_core_gene`
- `is_native_core`
- `is_anchored_core`
- `proc_Biosynthesis`
- `proc_Esterification`
- `proc_Excretion`
- `proc_Uptake`

如果你的后续 benchmark 想完全隔离 held-out 种子，这一步是必须的。


In [7]:
def get_leak_feature_indices(feature_cols, process_order):
    if feature_cols is None:
        return []

    leak_cols = ["is_core_gene", "is_native_core", "is_anchored_core"]
    leak_cols += [f"proc_{p}" for p in process_order]

    idxs = [feature_cols.index(c) for c in leak_cols if c in feature_cols]
    return sorted(set(idxs))

leak_feature_indices = get_leak_feature_indices(feature_cols, process_order)
print("leak_feature_indices =", leak_feature_indices)
if feature_cols is not None:
    print("leak feature names =", [feature_cols[i] for i in leak_feature_indices])


leak_feature_indices = []
leak feature names = []


In [8]:
def build_no_leak_bundle_for_split(split_id, split_sub_df, bundle, gene2idx, leak_feature_indices):
    new_bundle = copy.deepcopy(bundle)
    data = new_bundle["data"]

    # 兼容 supervised_mask / train_mask
    if hasattr(data, "supervised_mask"):
        mask = data.supervised_mask.clone()
    elif hasattr(data, "train_mask"):
        mask = data.train_mask.clone()
        data.supervised_mask = mask.clone()
    else:
        mask = (data.y >= 0)
        data.supervised_mask = mask.clone()

    y = data.y.clone()
    x = data.x.clone()

    heldout_genes = (
        split_sub_df.loc[split_sub_df["set_type"] == "heldout", "ENSGID"]
        .astype(str).drop_duplicates().tolist()
    )

    held_idx = [gene2idx[g] for g in heldout_genes if g in gene2idx]

    # 1) 从监督种子中移除
    if len(held_idx) > 0:
        idx_tensor = torch.tensor(held_idx, dtype=torch.long)
        mask[idx_tensor] = False

        # 2) 标签设为未知
        y[idx_tensor] = -1

        # 3) 清理泄漏特征
        if len(leak_feature_indices) > 0:
            feat_idx_tensor = torch.tensor(leak_feature_indices, dtype=torch.long)
            x[idx_tensor[:, None], feat_idx_tensor[None, :]] = 0.0

    data.supervised_mask = mask
    data.train_mask = mask.clone()  # 为兼容旧训练代码
    data.y = y
    data.x = x

    new_bundle["data"] = data
    new_bundle["benchmark_info"] = {
        "split_id": split_id,
        "heldout_genes": heldout_genes,
        "n_heldout_in_graph": len(held_idx),
        "leak_feature_indices": leak_feature_indices,
        "leak_feature_names": [feature_cols[i] for i in leak_feature_indices] if feature_cols is not None else []
    }
    return new_bundle


In [9]:
# ==============================
# 为每个 split 保存 Deep_GLOC no-leak bundle
# ==============================
bundle_summary_rows = []

for split_id, sub in split_long_df.groupby("split_id"):
    proc = sub["process"].iloc[0]
    out_subdir = os.path.join(PYG_SPLIT_DIR, split_id)
    os.makedirs(out_subdir, exist_ok=True)

    new_bundle = build_no_leak_bundle_for_split(
        split_id=split_id,
        split_sub_df=sub,
        bundle=bundle,
        gene2idx=gene2idx,
        leak_feature_indices=leak_feature_indices
    )

    out_pt = os.path.join(out_subdir, "pyg_data_no_leak.pt")
    torch.save(new_bundle, out_pt)

    info = new_bundle["benchmark_info"]
    data2 = new_bundle["data"]

    bundle_summary_rows.append({
        "split_id": split_id,
        "process": proc,
        "n_supervised_after_mask": int(data2.supervised_mask.sum().item()),
        "n_heldout_in_graph": info["n_heldout_in_graph"],
        "pt_file": out_pt
    })

bundle_summary_df = pd.DataFrame(bundle_summary_rows)
bundle_summary_path = os.path.join(SUMMARY_DIR, "pyg_split_bundle_summary.tsv")
bundle_summary_df.to_csv(bundle_summary_path, sep="\t", index=False)

print("saved:", bundle_summary_path)
display(bundle_summary_df.head())


saved: D:\phd cases\metabolic analysis\code\Deep_GLOC\run_GAT_new\masked_seed_benchmark\summary\pyg_split_bundle_summary.tsv


,split_id,process,n_supervised_after_mask,n_heldout_in_graph,pt_file
0,Biosynthesis__split_001,Biosynthesis,35,4,D:\phd cases\metabolic analysis\code\Deep_GLOC...
1,Biosynthesis__split_002,Biosynthesis,35,4,D:\phd cases\metabolic analysis\code\Deep_GLOC...
2,Biosynthesis__split_003,Biosynthesis,35,4,D:\phd cases\metabolic analysis\code\Deep_GLOC...
3,Biosynthesis__split_004,Biosynthesis,35,4,D:\phd cases\metabolic analysis\code\Deep_GLOC...
4,Biosynthesis__split_005,Biosynthesis,35,4,D:\phd cases\metabolic analysis\code\Deep_GLOC...


## 输出给师弟的文件

### 统一 split 文件
- `masked_seed_splits_long.tsv`
- `masked_seed_splits_meta.tsv`
- `splits/<split_id>/train_seeds.tsv`
- `splits/<split_id>/heldout_seeds.tsv`

### Deep_GLOC 专用文件
- `pyg_split_bundles/<split_id>/pyg_data_no_leak.pt`

建议分工：
- 你自己用 `pyg_data_no_leak.pt` 跑 Deep_GLOC；
- 师弟用同一个 `split_id` 下的 `train_seeds.tsv / heldout_seeds.tsv` 跑 RWR 和 DeepWalk；
- 最后统一汇总到同一张 benchmark 表。

## 下一步
下一本 notebook 可以继续做：
1. Deep_GLOC 批量运行每个 split；
2. 统一计算 Recall@K / MRR / median rank / AUROC / AUPRC；
3. 和 RWR / DeepWalk 的结果汇总比较。
